# Volume 14, Chapter 13: Random groups and the Cohen–Lenstra heuristics (Python companion)

Exercises first, solutions at the end. We compute the Cohen–Lenstra measure exactly, compare it with cokernels of random matrices over $\mathbb Z/p^e$, check the moment formula of Theorem 13.5, and test the conjecture on the class numbers of imaginary quadratic fields. The computer science application: **how many extra relations a factoring algorithm must collect** before its linear algebra over $\mathbb F_2$ is guaranteed to find a dependency, which is the $p$-rank law of Exercise 13.2 in disguise. Requires `numpy`.

In [ ]:
import math, random, itertools
from collections import Counter
import numpy as np

def c_inf(p, terms=200):
    return math.prod(1 - p ** -i for i in range(1, terms))

def partitions(n, maxpart=None):
    # partitions of n as non-increasing tuples
    maxpart = n if maxpart is None else maxpart
    if n == 0: yield (); return
    for k in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - k, k): yield (k,) + rest

def conj(lam):
    # conjugate type r_k = #{i : lam_i >= k}
    return [sum(1 for a in lam if a >= k) for k in range(1, (lam[0] if lam else 0) + 1)]

def aut(lam, p):
    # |Aut G| for G = prod Z/p^lam_i (Lemma 13.2)
    r = conj(lam); m = Counter(lam)
    return p ** sum(x * x for x in r) * math.prod(math.prod(1 - p ** -i for i in range(1, mi + 1)) for mi in m.values())

def mu_cl(lam, p):
    return c_inf(p) / aut(lam, p)

## Exercises

**P13.1 (★) The measure.** For $p=2,3,5$, list the groups of order $p^k$, $k\le 4$, with their Cohen–Lenstra probabilities. Check numerically that $\sum_{k\le K}\sum_{|G|=p^k}\mu_{CL}(G)\to1$ as $K$ grows, and compute the probability that $X$ is cyclic against Exercise 13.1.

**P13.2 (★★) Random cokernels.** Draw $12\,000$ uniform random $8\times8$ matrices over $\mathbb Z/27$, compute their cokernels by Smith normal form, and compare the frequencies with the exact Friedman–Washington probabilities (Theorem 13.6, with $n=8$, $e=3$) and with $\mu_{CL}$.

**P13.3 (★★) Moments.** With the same sample, estimate $\mathbb E\,\#\operatorname{Sur}(\operatorname{cok}M,G)$ for $G=\mathbb Z/3$, $\mathbb Z/9$, $(\mathbb Z/3)^2$ and compare with $\prod_{i<r}(1-3^{i-8})$ (Theorem 13.5). Why is the estimate for $(\mathbb Z/3)^2$ noisier?

**P13.4 (★★) Class numbers.** Compute $h(-d)$ for all $d\le 2\cdot10^5$ by counting reduced forms, keep the fundamental discriminants, and compare the proportions of $p\mid h(-d)$ for $p=3,5,7,11$ with $1-c_\infty(p)$. Then compare $\mathbb P(9\mid h)$ with the Cohen–Lenstra prediction.

**P13.5 (★★, CS application) Extra relations in the quadratic sieve.** The quadratic sieve and index calculus collect vectors in $\mathbb F_2^n$ (exponent vectors mod $2$ over a factor base of $n$ primes) and need a non-trivial linear dependency. Model the vectors as uniform random. (a) Compute the law of the corank of a random $n\times n$ matrix over $\mathbb F_2$ for $n=64$ by simulation (bitset Gaussian elimination) and compare with Exercise 13.2. (b) With $n+k$ random vectors, the dependencies form a space of dimension at least $k$; compute the probability that the vectors span $\mathbb F_2^n$, and explain the practice of collecting about $n+10$ to $n+20$ relations.

## Solutions

### P13.1

In [ ]:
for p in (2, 3, 5):
    print(f"p = {p}:  c_inf = {c_inf(p):.4f}")
    for k in range(5):
        print("   order p^%d:" % k, ", ".join(f"{lam}: {mu_cl(lam, p):.5f}" for lam in partitions(k)))
    for K in (2, 5, 10, 20):
        tot = sum(mu_cl(lam, p) for k in range(K + 1) for lam in partitions(k))
        print(f"   mass of |G| <= p^{K}: {tot:.6f}")
    cyc = sum(mu_cl((a,) if a else (), p) for a in range(60))
    print(f"   P(cyclic) = {cyc:.5f}   formula c_inf (1 + p/(p-1)^2) = {c_inf(p) * (1 + p / (p - 1) ** 2):.5f}")

For $p=3$ the trivial group has probability $0.5601$, $\mathbb Z/3$ half of that, and $(\mathbb Z/3)^2$ only $0.0117$, because it has $48$ automorphisms. The mass up to order $p^{20}$ is $1$ to six decimals: the measure is a probability (Corollary 13.7), and its tail is tiny since $|\operatorname{Aut}G|\ge c_\infty^e p^{\sum r_k^2}$ grows fast. For $p=3$, $98\%$ of the mass sits on cyclic groups.

### P13.2

In [ ]:
def cokernel(M, p, e):
    # Smith normal form over Z/p^e: returns the type of the cokernel (parts a with Z/p^a, a >= 1)
    q = p ** e; A = [list(map(int, r)) for r in M]; n = len(A); parts = []
    def val(x):
        x %= q
        if x == 0: return e
        v = 0
        while x % p == 0: x //= p; v += 1
        return v
    for k in range(n):
        best = None
        for i in range(k, n):
            for j in range(k, n):
                v = val(A[i][j])
                if best is None or v < best[0]: best = (v, i, j)
        v, i, j = best
        if v >= e: parts += [e] * (n - k); break
        A[k], A[i] = A[i], A[k]
        for r in A: r[k], r[j] = r[j], r[k]
        u = pow(A[k][k] // p ** v, -1, q)
        for i2 in range(k + 1, n):
            f = (A[i2][k] // p ** v) * u % q
            if f: A[i2] = [(x - f * y) % q for x, y in zip(A[i2], A[k])]
        for j2 in range(k + 1, n):
            f = (A[k][j2] // p ** v) * u % q
            if f:
                for r in range(n): A[r][j2] = (A[r][j2] - f * A[r][k]) % q
        parts.append(v)
    return tuple(sorted((a for a in parts if a > 0), reverse=True))

p, e, n, S = 3, 3, 8, 12000
rng = np.random.default_rng(13)
obs = Counter(cokernel(rng.integers(0, p ** e, (n, n)), p, e) for _ in range(S))

def fw(lam, p, n):
    r = len(lam)
    return math.prod(1 - p ** (i - n) for i in range(r)) * math.prod(1 - p ** -j for j in range(1, n + 1)) / aut(lam, p)

print(f"{'group':12s} {'observed':>9s} {'FW n=8':>9s} {'mu_CL':>9s}")
for lam in [(), (1,), (2,), (1, 1), (2, 1), (1, 1, 1), (2, 2)]:
    print(f"{str(lam):12s} {obs[lam] / S:9.4f} {fw(lam, p, n):9.4f} {mu_cl(lam, p):9.4f}")
print("exponent 27 seen (truncated, excluded from Theorem 13.6):", sum(c for lam, c in obs.items() if lam and lam[0] == e) / S)

The observed frequencies agree with the exact Friedman–Washington values within sampling error (about $\pm0.005$ for the trivial group), and already for $n=8$ these are within $0.001$ of $\mu_{CL}$: the factor $\prod_{j\le n}(1-p^{-j})$ converges geometrically fast. About $5\%$ of the cokernels have exponent $27$; they are the truncations of larger cyclic factors that $\mathbb Z/27$ cannot see, which is why Theorem 13.6 only speaks about exponent dividing $p^{e-1}$.

### P13.3

In [ ]:
def n_sur(lam, G, p):
    # number of surjections from prod Z/p^lam onto G = prod Z/p^G, by Mobius inversion over subgroups is heavy;
    # instead count homomorphisms onto: Hom(A,G) minus those landing in a maximal subgroup. For the three small G we do it directly.
    def hom(A, B):
        return p ** sum(min(a, b) for a in A for b in B)
    if G == (1,): return hom(lam, (1,)) - 1
    if G == (2,): return hom(lam, (2,)) - hom(lam, (1,))
    if G == (1, 1): return hom(lam, (1, 1)) - (p + 1) * hom(lam, (1,)) + p
    raise ValueError

for G, r in [((1,), 1), ((2,), 1), ((1, 1), 2)]:
    vals = [n_sur(lam, G, p) for lam, c in obs.items() for _ in range(c)]
    est = np.mean(vals); se = np.std(vals) / math.sqrt(S)
    print(f"G = {G}:  estimate {est:.4f} +- {se:.4f}   Theorem 13.5: {math.prod(1 - p ** (i - n) for i in range(r)):.4f}")

A homomorphism onto $\mathbb Z/9$ is a homomorphism to $\mathbb Z/9$ that does not land in $3\mathbb Z/9\cong\mathbb Z/3$; onto $(\mathbb Z/3)^2$, we subtract those landing in one of the $p+1$ lines and add back the zero map counted $p+1$ times instead of once. The three estimates match $\prod_{i<r}(1-3^{i-8})\approx1$ within their standard errors. The one for $(\mathbb Z/3)^2$ is noisier because $\#\operatorname{Sur}(X,(\mathbb Z/3)^2)$ is $0$ most of the time and $|G|^{\operatorname{rank}X}$-large when $X$ has rank $2$ or more: the second moment grows like $p^{r^2}$-type quantities, which is exactly why the uniqueness theorem needs the bound $C|\wedge^2G|$.

### P13.4

In [ ]:
N = 200_000
h = np.zeros(N + 1, dtype=np.int64)
amax = int(math.isqrt(N // 3)) + 1
for a in range(1, amax + 1):
    for b in range(-a + 1, a + 1):
        c0 = max(a, (b * b + 4 * a - 1) // (4 * a))   # 4ac - b^2 >= 1 and c >= a
        cmax = (N + b * b) // (4 * a)
        if cmax < c0: continue
        c = np.arange(c0, cmax + 1)
        if b < 0: c = c[c > a]                         # reduced: b >= 0 when a == c
        d = 4 * a * c - b * b
        np.add.at(h, d, 1)
# squarefree sieve and fundamental discriminants -d (d > 4)
sf = np.ones(N + 1, bool)
for q in range(2, math.isqrt(N) + 1): sf[q * q::q * q] = False
dd = np.arange(N + 1)
fund = ((dd % 4 == 3) & sf) | ((dd % 4 == 0) & np.isin((dd // 4) % 4, [1, 2]) & sf[dd // 4])
fund[:5] = False
H = h[fund]
print("fundamental discriminants:", fund.sum(), "  check h(-23) =", h[23], ", h(-56) =", h[56], ", h(-163) =", h[163])
for p in (3, 5, 7, 11):
    print(f"p = {p:2d}:  P(p | h) = {np.mean(H % p == 0):.4f}   Cohen-Lenstra 1 - c_inf = {1 - c_inf(p):.4f}")
cl9 = 1 - sum(mu_cl(lam, 3) for k in range(2) for lam in partitions(k))
print(f"P(9 | h) = {np.mean(H % 9 == 0):.4f}   Cohen-Lenstra P(|G| >= 9) = {cl9:.4f}")

The proportions are below the predictions, by about $0.05$ for $p=3$ and less for larger $p$, and they keep creeping up as $d$ grows (Figure 13.2 runs the same computation to $10^7$): the secondary term of size $X^{-1/6}$ in the Davenport–Heilbronn count makes $3$ converge slowly. Note that $h(-d)$ counts reduced forms of discriminant $-d$, including non-primitive ones for non-fundamental $d$; for fundamental $d$ all forms are primitive, so $h$ is the class number. $\mathbb P(9\mid h)$ compares to $\mu_{CL}(|G|\ge9)$, the groups of order at least $9$, which is about $0.16$; here we are again below, for the same reason.

### P13.5

In [ ]:
def rank_f2(rows):
    # rank of a list of n-bit integers over F_2 (xor basis)
    basis = {}
    r = 0
    for v in rows:
        while v:
            top = v.bit_length() - 1
            if top in basis: v ^= basis[top]
            else: basis[top] = v; r += 1; break
    return r

random.seed(2)
n, T = 64, 20000
cor = Counter(n - rank_f2([random.getrandbits(n) for _ in range(n)]) for _ in range(T))
c2 = c_inf(2)
print("corank  observed   limit law  c_inf 2^(-r^2) / prod (1-2^-i)^2")
for r in range(4):
    lim = c2 * 2 ** (-r * r) / math.prod((1 - 2 ** -i) ** 2 for i in range(1, r + 1))
    print(f"{r:6d}  {cor[r] / T:9.4f}  {lim:9.4f}")

print("\nk extra vectors: P(n+k random vectors span F_2^n) = prod_{i>k} (1 - 2^-i)")
for k in (0, 1, 2, 5, 10, 20):
    exact = math.prod(1 - 2.0 ** -i for i in range(k + 1, n + k + 1))
    sim = np.mean([rank_f2([random.getrandbits(n) for _ in range(n + k)]) == n for _ in range(3000)])
    print(f"k = {k:2d}:  exact {exact:.6f}   simulated {sim:.4f}")

(a) A random $64\times64$ matrix over $\mathbb F_2$ is invertible with probability $0.2888=c_\infty(2)$, has corank $1$ with probability $0.5776$, corank $2$ with $0.1284$: exactly the $2$-rank law of the Cohen–Lenstra measure, as Exercise 13.2 shows. (b) The $n+k$ vectors span $\mathbb F_2^n$ with probability $\prod_{i=k+1}^{n+k}(1-2^{-i})\approx1-2^{-k}$: an $(n+k)\times n$ matrix of rank $n$ has a kernel of dimension exactly $k$. In the quadratic sieve one needs a dependency that gives a non-trivial factor, and each dependency succeeds with probability about $\frac12$; so with $k$ extra relations the failure probability is about $2^{-k}$, and $k=10$ to $20$ makes failure practically impossible. Real exponent vectors are sparse and not uniform, so this is a heuristic model, but it is the one practitioners use to size the extra relations.